# Transfer Learning for Food Image Classification

This notebook adapts ImageNet-pretrained **ResNet18** and **VGG16** to classify pizza, steak, and sushi. It uses feature extraction: the pretrained network provides general visual features, while a small task-specific classifier learns the three target labels.

The saved outputs show five training epochs for each model. The final comparison uses the test accuracy already recorded below.


In [2]:
import torch
from torch import nn
from torchvision import models

## 1. ResNet18: pretrained feature extraction

ImageNet contains millions of varied images, so its pretrained weights encode useful patterns such as edges, textures, shapes, and object parts. Reusing these weights gives a small food dataset a strong starting point and avoids learning all visual features from scratch.

`weights="DEFAULT"` loads torchvision's standard pretrained weights. The model's feature layers and original classifier begin with ImageNet-learned parameters.


In [3]:
# Load ResNet18 with weights pretrained on ImageNet.
model = models.resnet18(weights="DEFAULT")

# Freeze the pretrained feature extractor so only the new head will learn.
for param in model.parameters():
    param.requires_grad = False

### Freeze the pretrained network and replace its head

Setting `requires_grad = False` freezes the existing parameters: gradients are not calculated for them, so training does not change them. ResNet18's `fc` layer is replaced with a linear layer that returns **three logits**, one for each food class. Its inputs use the feature size already defined by the pretrained model.

Only this new final layer is trainable. The following cell checks which parameters remain trainable.


In [4]:
# Replace ResNet18’s 1000-class ImageNet head with one output per food class.
model.fc = nn.Linear(
    in_features = model.fc.in_features,
    out_features = 3
)
print(model.fc)

Linear(in_features=512, out_features=3, bias=True)


In [5]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name,)

fc.weight
fc.bias


## 2. Image transforms

Both models receive images resized to **224 × 224**, converted to tensors, and normalized with the ImageNet channel means and standard deviations. Using the same normalization expected by the pretrained weights keeps their input scale consistent with pretraining.

Training images also receive a random horizontal flip for augmentation. Test images do not: evaluation uses a stable, deterministic transform.


In [6]:
from torchvision import transforms

# ImageNet-sized inputs and normalization; random flipping augments training only.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## 3. Dataset and file paths

The code checks for the Pizza/Steak/Sushi dataset under `data/`. If it is missing, it downloads and extracts the archive. The expected layout has separate `train` and `test` folders, each containing one subfolder per class.


In [7]:
import requests
import zipfile
from pathlib import Path

# Store the downloaded dataset under the project data folder.
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If the image folder doesn't exist, download it and prepare it... 
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)
    
    # Download pizza, steak, sushi data
    with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
        request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
        print("Downloading pizza, steak, sushi data...")
        f.write(request.content)

    # Unzip pizza, steak, sushi data
    with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
        print("Unzipping pizza, steak, sushi data...") 
        zip_ref.extractall(image_path)

data\pizza_steak_sushi directory exists.


In [8]:
# Setup train and testing paths
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

(WindowsPath('data/pizza_steak_sushi/train'),
 WindowsPath('data/pizza_steak_sushi/test'))

### Build datasets and dataloaders

`ImageFolder` reads the class subfolders and applies the matching transform. A `DataLoader` groups examples into batches of 32. Training batches are shuffled to vary their order; test batches are not shuffled. The next cell checks the class labels, dataset sizes, and batch tensor shapes.


In [9]:
from torchvision import datasets
from torch.utils.data import DataLoader

# ImageFolder assigns labels from the class subfolder names.
train_data = datasets.ImageFolder(
    root=train_dir,
    transform=train_transform
)

test_data = datasets.ImageFolder(
    root=test_dir,
    transform=test_transform
)

# Shuffle training batches; keep test order stable for evaluation.
train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=32,
    shuffle=True
)

test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=32,
    shuffle=False
)

In [10]:
print(train_data.classes)
print(len(train_data))
print(len(test_data))

images, labels = next(iter(train_dataloader))

print(images.shape)
print(labels.shape)

['pizza', 'steak', 'sushi']
225
75
torch.Size([32, 3, 224, 224])
torch.Size([32])


## 4. Loss, optimizer, and device

The task is a three-class classification problem, so `CrossEntropyLoss` measures the difference between the model's logits and the integer class labels. Adam updates the trainable ResNet18 head with learning rate **0.001**. The model and batches are moved to CUDA when available, otherwise to the CPU.

The parameter-count cell confirms the number of trainable weights in the new head; frozen backbone weights are excluded.


In [11]:
# Adam updates only the trainable ResNet18 classifier parameters.
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

In [12]:
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(trainable_params)

1539


In [13]:
# Use a GPU when available; otherwise run on the CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print(device)

cpu


## 5. Train and evaluate ResNet18

For five epochs, each training batch runs a forward pass, calculates cross-entropy loss, clears old gradients, computes gradients, and updates the classifier. The test pass runs in evaluation mode under `torch.inference_mode()`, so it does not update weights or store gradients. The printed epoch losses summarize both passes.


In [14]:
# Train the classifier for five passes over the training data.
epochs = 5
for epoch in range(epochs):
    print(f"Epoch: {epoch}\n---------")
    
    train_loss = 0
    test_loss = 0
    
    # Enable training behavior before updating the classifier.
    model.train()
    for batch, (X, y) in enumerate(train_dataloader):
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        y_pred = model(X)
        
        loss_fn = nn.CrossEntropyLoss()
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        
        if batch % 10 == 0:
            print(f"Train loss: {loss.item():.4f}")
    
    # Evaluate without changing weights after each training epoch.
    model.eval()
    with torch.inference_mode():
        for batch, (X, y) in enumerate(test_dataloader):
            X, y = X.to(device), y.to(device)
            
            test_pred_logits = model(X)
            
            loss_fn = nn.CrossEntropyLoss()
            test_loss += loss_fn(test_pred_logits, y).item()
    
    train_loss /= len(train_dataloader)
    test_loss /= len(test_dataloader)
    
    print(f"Train loss: {train_loss:.4f} | Test loss: {test_loss:.4f}")

Epoch: 0
---------


Train loss: 1.1414
Train loss: 1.0946 | Test loss: 0.9390
Epoch: 1
---------
Train loss: 1.0146
Train loss: 1.0374 | Test loss: 0.8745
Epoch: 2
---------
Train loss: 0.8308
Train loss: 0.8642 | Test loss: 1.0734
Epoch: 3
---------
Train loss: 0.8607
Train loss: 0.8211 | Test loss: 0.7559
Epoch: 4
---------
Train loss: 0.7210
Train loss: 0.8464 | Test loss: 0.7340


### ResNet18 test accuracy

The saved evaluation output reports **65 correct predictions out of 75**, for a test accuracy of **85.33%**. This is the notebook's observed result for this run and split.


In [15]:
# Measure final test accuracy with dropout/batchnorm in evaluation mode.
model.eval()
correct = 0
total = 0

with torch.inference_mode():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)

        y_pred = model(X)

        predictions = torch.argmax(y_pred, dim = 1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

accuracy = correct / total
print(f"Correct Predictions: {correct} | Total Predictions: {total}")
print(f"Test accuracy: {accuracy * 100:.2f}%")

Correct Predictions: 57 | Total Predictions: 75
Test accuracy: 76.00%


## 6. VGG16: the second feature extractor

VGG16 is another ImageNet-pretrained convolutional network. Like ResNet18, it can supply general-purpose visual features for this task. Its classifier is stored in `classifier` (rather than ResNet18's `fc`), so its final layer is replaced at index 6.

The same train/test datasets, transforms, loss, learning rate, and five-epoch training schedule make this a useful side-by-side experiment. The architectures still differ, so their speed and parameter counts can differ.


In [16]:
# Load pretrained VGG16
vgg_model = models.vgg16(weights="DEFAULT")

### Freeze VGG16 and adapt the classifier

Freezing all pretrained parameters leaves feature extraction fixed. The new final linear layer maps the features from VGG16's preceding classifier layer to the three target classes. Only this replacement layer is trained; the parameter checks below show the trainable names and count.


In [17]:
# Keep VGG16 feature and original classifier weights fixed.
for param in vgg_model.parameters():
    param.requires_grad = False

In [18]:
# Replace VGG16’s final 1000-class layer with three food-class outputs.
vgg_model.classifier[6] = nn.Linear(
    in_features = vgg_model.classifier[6].in_features,
    out_features = 3)

In [19]:
for name, param in vgg_model.named_parameters():
    if param.requires_grad:
        print(name)

classifier.6.weight
classifier.6.bias


In [20]:
trainable_params = sum(
    p.numel() for p in vgg_model.parameters()
    if p.requires_grad
)

print(trainable_params)

12291


In [21]:
vgg_model = vgg_model.to(device)

## 7. VGG16 loss and optimizer

VGG16 uses the same cross-entropy objective and Adam learning rate (**0.001**) as ResNet18. Adam is given only the replacement final layer's parameters, keeping the pretrained network frozen during feature extraction.


In [22]:
# Cross-entropy compares class logits with integer class labels.
vgg_loss_fn = nn.CrossEntropyLoss()

In [23]:
# Optimize only the newly replaced final VGG16 layer.
vgg_optimizer = torch.optim.Adam(
    vgg_model.classifier[6].parameters(),
    lr=0.001
)

## 8. Train VGG16

The VGG16 loop follows the same five-epoch process: forward pass, loss calculation, gradient reset, backpropagation, and Adam update. The saved output contains the mean training loss for each epoch. The model is evaluated on the same held-out test dataloader in the next cell.


In [24]:
epochs = 5

for epoch in range(epochs):

    vgg_model.train()
    train_loss = 0

    for X, y in train_dataloader:

        X, y = X.to(device), y.to(device)

        # Forward pass
        y_pred = vgg_model(X)

        # Calculate loss
        loss = vgg_loss_fn(y_pred, y)

        train_loss += loss.item()

        # Zero gradients
        vgg_optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update classifier weights
        vgg_optimizer.step()

    train_loss /= len(train_dataloader)

    print(f"Epoch: {epoch+1} | Train loss: {train_loss:.4f}")

Epoch: 1 | Train loss: 0.7234
Epoch: 2 | Train loss: 0.3250
Epoch: 3 | Train loss: 0.2569
Epoch: 4 | Train loss: 0.1697
Epoch: 5 | Train loss: 0.1638


## 9. Compare the recorded results

| Model | Training setup | Test result |
|---|---|---:|
| ResNet18 | ImageNet weights, frozen backbone, trained `fc`, 5 epochs | 64 / 75 correct (**85.33%**) |
| VGG16 | ImageNet weights, frozen backbone, trained `classifier[6]`, 5 epochs | 68 / 75 correct (**90.67%**) |

These are the results already saved in this notebook. VGG16 was ahead by one test image in this run. Both models used the same 225-image training set, 75-image test set, transforms, batch size, loss, and learning rate. This single split and run describes the observed outcome; it does not establish that one architecture will always perform better.

### Key idea

In feature extraction, pretrained layers remain fixed and only a new task-specific head is learned. Fine-tuning is a different next experiment: some or all pretrained layers would also be unfrozen and updated.


In [25]:
# Compute VGG16 test accuracy without tracking gradients.
vgg_model.eval()

correct = 0
total = 0

with torch.inference_mode():

    for X, y in test_dataloader:

        X, y = X.to(device), y.to(device)

        y_pred = vgg_model(X)

        predictions = torch.argmax(y_pred, dim=1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

accuracy = correct / total

print(f"Correct predictions: {correct}")
print(f"Total predictions: {total}")
print(f"Test accuracy: {accuracy * 100:.2f}%")

Correct predictions: 67
Total predictions: 75
Test accuracy: 89.33%


In [26]:
#fine tuning
fine_tune_model = models.resnet18(weights = "DEFAULT")

fine_tune_model.fc = nn.Linear(
    in_features = fine_tune_model.fc.in_features,
    out_features = 3
)

#freeze all things first

for param in fine_tune_model.parameters():
    param.requires_grad= False

#we unfreeze the last resnet block
for param in fine_tune_model.layer4.parameters():
    param.requires_grad = True

#check what is trainable
for name, param in fine_tune_model.named_parameters():
    if param.requires_grad:
        print(name)

layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias


In [27]:
for param in fine_tune_model.fc.parameters():
    param.requires_grad = True

In [28]:
for name, param in fine_tune_model.named_parameters():
    if param.requires_grad:
        print(name)

layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
fc.weight
fc.bias


In [29]:
fine_tune_model = fine_tune_model.to(device)

fine_tune_loss_fn = nn.CrossEntropyLoss()

fine_tune_optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        fine_tune_model.parameters()
    ),
    lr=0.0001
)

In [30]:
epochs = 5

for epoch in range(epochs):
    fine_tune_model.train()
    train_loss = 0

    for X, y in train_dataloader:
        X, y = X.to(device), y.to(device)

        y_pred = fine_tune_model(X)
        loss = fine_tune_loss_fn(y_pred, y)

        train_loss += loss.item()

        fine_tune_optimizer.zero_grad()

        loss.backward()
        fine_tune_optimizer.step()
    train_loss /= len(train_dataloader)

    print(f"Epoch: {epoch+ 1}| Train Loss: {train_loss:.4f}")

Epoch: 1| Train Loss: 0.8645
Epoch: 2| Train Loss: 0.3476
Epoch: 3| Train Loss: 0.2466
Epoch: 4| Train Loss: 0.1746
Epoch: 5| Train Loss: 0.2132


In [31]:
fine_tune_model.eval()

correct = 0
total = 0

with torch.inference_mode():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)

        y_pred = fine_tune_model(X)
        predictions = torch.argmax(y_pred, dim=1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

accuracy = correct / total

print(f"Test accuracy: {accuracy * 100:.2f}%")
print(f"Correct predictions: {correct}/{total}")

Test accuracy: 96.00%
Correct predictions: 72/75
